In [1]:
!pip install transformers
!pip install pytorch
!pip install torch

  Using cached pytorch-1.0.2.tar.gz (689 bytes)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build pytorch


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [37 lines of output]
  Traceback (most recent call last):
    File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
      ~~~~^^
    File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                               ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
    File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 280, in build_wheel
      return _build_backend().build_wheel(
             ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
          wheel_directory, config_settings, metadata_directory
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      )
      ^
    File "C:\Users\Administrator\AppData\Local\Temp\pip-build-env-7nmgkpvb\overlay\Lib\site-packages\setuptools\

In [2]:
import torch
from transformers import pipeline

# NLP tasks

### semantic analysis

In [3]:
classifier = pipeline("text-classification")
result = classifier("I am not happy with the last mission impossible movie")
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997031092643738}]


In [4]:
classi = pipeline("sentiment-analysis", model="SamLowe/roberta-base-go_emotions")
task_list = ["i really like autoencoder, best models for anamoly detection",
            "i am not sure if can actually evaluate llms",
            "i hate long meetings"]
result = classi(task_list)
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'admiration', 'score': 0.6213876008987427}, {'label': 'confusion', 'score': 0.9028298854827881}, {'label': 'anger', 'score': 0.7424210906028748}]


# tokenization

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DistilBertTokenizer,
    DistilBertForSequenceClassification
)

In [6]:
model_name1 = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer1 = DistilBertTokenizer.from_pretrained(model_name1)
mymodel1 = DistilBertForSequenceClassification.from_pretrained(model_name1)

classifier = pipeline("sentiment-analysis", model= mymodel1, tokenizer = tokenizer1)
res = classifier("I was not so happy with barbie movie")
print(res)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9996719360351562}]


In [7]:
model_name2 = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer2 = AutoTokenizer.from_pretrained(model_name2)
mymodel2 = AutoModelForSequenceClassification.from_pretrained(model_name2)

classifier = pipeline("sentiment-analysis", model= mymodel2, tokenizer = tokenizer2)
res = classifier("I was not so happy with barbie movie")
print(res)

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': '2 stars', 'score': 0.46677812933921814}]


# fine tuning

In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset('imdb')

### preprocess the data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation = True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

### set up the training arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',              # Output directory
    eval_strategy="epoch",               # Evaluate every epoch
    learning_rate=2e-5,                  # Learning rate
    per_device_train_batch_size=16,      # Batch size for training
    per_device_eval_batch_size=16,       # Batch size for evaluation
    num_train_epochs=1,                  # Number of training epochs
    weight_decay=0.01                    # Strength of weight decay
)

training_args

### initialize the model

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test']
)

### train the model

In [ ]:
trainer.train()

### evaluate the model

In [ ]:
results = trainer.evaluate()
print(results)

### save the fine tuned model

In [ ]:
model.save_pretrained('./fine-tuned-model')
tokenizer.save_pretrained('./fine-tuned-model')